In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# AER Dataset Exploration\n",
    "## Abductive Event Reasoning - Data Analysis and Visualization\n",
    "\n",
    "This notebook provides an interactive exploration of the AER dataset."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Setup and Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "from collections import Counter, defaultdict\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "\n",
    "from src.data.loader import AERDataLoader, AERInstance\n",
    "from src.data.preprocessor import TextPreprocessor\n",
    "from src.utils.helpers import parse_answer\n",
    "\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set plot style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "plt.rcParams['font.size'] = 10\n",
    "\n",
    "print(\"✓ Imports successful\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Load Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load data\n",
    "data_dir = '../data/raw'\n",
    "loader = AERDataLoader(data_dir)\n",
    "instances = loader.load()\n",
    "\n",
    "print(f\"📊 Loaded {len(instances)} instances\")\n",
    "print(f\"📄 Total documents: {sum(len(inst.docs) for inst in instances)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Basic Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get statistics\n",
    "stats = loader.get_statistics()\n",
    "\n",
    "# Create DataFrame for display\n",
    "stats_df = pd.DataFrame([\n",
    "    {'Metric': 'Total Instances', 'Value': stats['total_instances']},\n",
    "    {'Metric': 'Total Documents', 'Value': stats['total_documents']},\n",
    "    {'Metric': 'Avg Docs per Instance', 'Value': f\"{stats['avg_docs_per_instance']:.2f}\"},\n",
    "    {'Metric': 'Avg Document Length (chars)', 'Value': f\"{stats['avg_doc_length']:.0f}\"},\n",
    "    {'Metric': 'Max Document Length (chars)', 'Value': stats['max_doc_length']},\n",
    "    {'Metric': 'Min Document Length (chars)', 'Value': stats['min_doc_length']}\n",
    "])\n",
    "\n",
    "display(stats_df)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Sample Data Inspection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display first instance\n",
    "sample_inst = instances[0]\n",
    "\n",
    "print(\"=\" * 80)\n",
    "print(\"SAMPLE INSTANCE\")\n",
    "print(\"=\" * 80)\n",
    "print(f\"\\n📌 Topic ID: {sample_inst.topic_id}\")\n",
    "print(f\"\\n❓ Question:\\n{sample_inst.question}\")\n",
    "print(f\"\\n📝 Options:\")\n",
    "for key, value in sample_inst.get_options_dict().items():\n",
    "    print(f\"  {key}. {value}\")\n",
    "print(f\"\\n✅ Answer: {sample_inst.answer}\")\n",
    "print(f\"\\n📄 Number of Documents: {len(sample_inst.docs)}\")\n",
    "print(f\"\\nFirst Document:\")\n",
    "if sample_inst.docs:\n",
    "    first_doc = sample_inst.docs[0]\n",
    "    print(f\"  Title: {first_doc.get('title', 'N/A')}\")\n",
    "    print(f\"  Content (first 200 chars): {first_doc.get('content', '')[:200]}...\")\n",
    "print(\"=\" * 80)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Answer Distribution Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Count answer combinations\n",
    "answer_counts = Counter(inst.answer for inst in instances if inst.answer)\n",
    "\n",
    "# Create DataFrame\n",
    "answer_df = pd.DataFrame([\n",
    "    {'Answer': ans, 'Count': count, 'Percentage': f\"{(count/len(instances)*100):.2f}%\"}\n",
    "    for ans, count in answer_counts.most_common(20)\n",
    "])\n",
    "\n",
    "print(\"\\n📊 Top 20 Most Common Answers:\")\n",
    "display(answer_df)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize answer distribution\n",
    "top_15 = dict(answer_counts.most_common(15))\n",
    "\n",
    "fig = go.Figure([\n",
    "    go.Bar(\n",
    "        x=list(top_15.keys()),\n",
    "        y=list(top_15.values()),\n",
    "        text=list(top_15.values()),\n",
    "        textposition='auto',\n",
    "        marker_color='steelblue'\n",
    "    )\n",
    "])\n",
    "\n",
    "fig.update_layout(\n",
    "    title='Top 15 Answer Distributions',\n",
    "    xaxis_title='Answer Combination',\n",
    "    yaxis_title='Frequency',\n",
    "    height=500,\n",
    "    showlegend=False\n",
    ")\n",
    "\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Single vs Multiple Answer Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Count single vs multiple answers\n",
    "single_answers = sum(1 for inst in instances if inst.answer and ',' not in inst.answer)\n",
    "multiple_answers = sum(1 for inst in instances if inst.answer and ',' in inst.answer)\n",
    "\n",
    "# Pie chart\n",
    "fig = go.Figure(data=[\n",
    "    go.Pie(\n",
    "        labels=['Single Answer', 'Multiple Answers'],\n",
    "        values=[single_answers, multiple_answers],\n",
    "        hole=0.4,\n",
    "        marker_colors=['#3498db', '#e74c3c']\n",
    "    )\n",
    "])\n",
    "\n",
    "fig.update_layout(\n",
    "    title='Single vs Multiple Answer Distribution',\n",
    "    height=400\n",
    ")\n",
    "\n",
    "fig.show()\n",
    "\n",
    "print(f\"Single Answers: {single_answers} ({single_answers/len(instances)*100:.1f}%)\")\n",
    "print(f\"Multiple Answers: {multiple_answers} ({multiple_answers/len(instances)*100:.1f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Option Frequency Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Count how often each option appears in correct answers\n",
    "option_counts = defaultdict(int)\n",
    "\n",
    "for inst in instances:\n",
    "    if inst.answer:\n",
    "        for opt in parse_answer(inst.answer):\n",
    "            option_counts[opt] += 1\n",
    "\n",
    "# Create visualization\n",
    "options = ['A', 'B', 'C', 'D']\n",
    "counts = [option_counts.get(opt, 0) for opt in options]\n",
    "colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']\n",
    "\n",
    "fig = go.Figure([\n",
    "    go.Bar(\n",
    "        x=options,\n",
    "        y=counts,\n",
    "        text=counts,\n",
    "        textposition='auto',\n",
    "        marker_color=colors\n",
    "    )\n",
    "])\n",
    "\n",
    "fig.update_layout(\n",
    "    title='Option Frequency in Correct Answers',\n",
    "    xaxis_title='Option',\n",
    "    yaxis_title='Frequency',\n",
    "    height=400\n",
    ")\n",
    "\n",
    "fig.show()\n",
    "\n",
    "# Print percentages\n",
    "total = sum(counts)\n",
    "for opt, count in zip(options, counts):\n",
    "    print(f\"Option {opt}: {count} ({count/total*100:.1f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. \"Insufficient Information\" (Option D) Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Count instances with option D\n",
    "has_d_only = sum(1 for inst in instances if inst.answer == 'D')\n",
    "has_d_with_others = sum(1 for inst in instances if 'D' in inst.answer and ',' in inst.answer)\n",
    "no_d = sum(1 for inst in instances if inst.answer and 'D' not in inst.answer)\n",
    "\n",
    "categories = ['Only D\\n(Insufficient Info)', 'D with Other Options', 'No D']\n",
    "values = [has_d_only, has_d_with_others, no_d]\n",
    "\n",
    "fig = go.Figure([\n",
    "    go.Bar(\n",
    "        x=categories,\n",
    "        y=values,\n",
    "        text=values,\n",
    "        textposition='auto',\n",
    "        marker_color=['#e74c3c', '#f39c12', '#2ecc71']\n",
    "    )\n",
    "])\n",
    "\n",
    "fig.update_layout(\n",
    "    title='\"Insufficient Information\" (Option D) Distribution',\n",
    "    xaxis_title='Category',\n",
    "    yaxis_title='Count',\n",
    "    height=400\n",
    ")\n",
    "\n",
    "fig.show()\n",
    "\n",
    "print(f\"\\nOnly D (Insufficient): {has_d_only} ({has_d_only/len(instances)*100:.1f}%)\")\n",
    "print(f\"D with others: {has_d_with_others} ({has_d_with_others/len(instances)*100:.1f}%)\")\n",
    "print(f\"No D: {no_d} ({no_d/len(instances)*100:.1f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Document Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Document counts per instance\n",
    "doc_counts = [len(inst.docs) for inst in instances]\n",
    "\n",
    "# Create histogram\n",
    "fig = go.Figure([\n",
    "    go.Histogram(\n",
    "        x=doc_counts,\n",
    "        nbinsx=20,\n",
    "        marker_color='steelblue',\n",
    "        opacity=0.7\n",
    "    )\n",
    "])\n",
    "\n",
    "fig.update_layout(\n",
    "    title='Distribution of Document Counts per Instance',\n",
    "    xaxis_title='Number of Documents',\n",
    "    yaxis_title='Frequency',\n",
    "    height=400\n",
    ")\n",
    "\n",
    "fig.show()\n",
    "\n",
    "print(f\"\\nDocument Count Statistics:\")\n",
    "print(f\"  Min: {min(doc_counts)}\")\n",
    "print(f\"  Max: {max(doc_counts)}\")\n",
    "print(f\"  Mean: {np.mean(doc_counts):.2f}\")\n",
    "print(f\"  Median: {np.median(doc_counts):.0f}\")\n",
    "print(f\"  Std Dev: {np.std(doc_counts):.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Text Length Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Question lengths\n",
    "question_lengths = [len(inst.question.split()) for inst in instances]\n",
    "\n",
    "# Document lengths\n",
    "doc_lengths = []\n",
    "for inst in instances:\n",
    "    for doc in inst.docs:\n",
    "        content = doc.get('content', '')\n",
    "        doc_lengths.append(len(content.split()))\n",
    "\n",
    "# Option lengths\n",
    "option_lengths = []\n",
    "for inst in instances:\n",
    "    for opt in inst.get_options_list():\n",
    "        option_lengths.append(len(opt.split()))\n",
    "\n",
    "# Create subplots\n",
    "fig = make_subplots(\n",
    "    rows=1, cols=3,\n",
    "    subplot_titles=('Question Length', 'Document Length', 'Option Length')\n",
    ")\n",
    "\n",
    "fig.add_trace(\n",
    "    go.Histogram(x=question_lengths, name='Questions', marker_color='skyblue'),\n",
    "    row=1, col=1\n",
    ")\n",
    "\n",
    "fig.add_trace(\n",
    "    go.Histogram(x=doc_lengths, name='Documents', marker_color='lightcoral'),\n",
    "    row=1, col=2\n",
    ")\n",
    "\n",
    "fig.add_trace(\n",
    "    go.Histogram(x=option_lengths, name='Options', marker_color='lightgreen'),\n",
    "    row=1, col=3\n",
    ")\n",
    "\n",
    "fig.update_xaxes(title_text=\"Words\", row=1, col=1)\n",
    "fig.update_xaxes(title_text=\"Words\", row=1, col=2)\n",
    "fig.update_xaxes(title_text=\"Words\", row=1, col=3)\n",
    "\n",
    "fig.update_yaxes(title_text=\"Frequency\", row=1, col=1)\n",
    "\n",
    "fig.update_layout(\n",
    "    title_text='Text Length Distributions (in words)',\n",
    "    height=400,\n",
    "    showlegend=False\n",
    ")\n",
    "\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Print statistics\n",
    "print(\"Text Length Statistics (in words):\")\n",
    "print(\"\\nQuestions:\")\n",
    "print(f\"  Min: {min(question_lengths)}\")\n",
    "print(f\"  Max: {max(question_lengths)}\")\n",
    "print(f\"  Mean: {np.mean(question_lengths):.1f}\")\n",
    "print(f\"  Median: {np.median(question_lengths):.0f}\")\n",
    "\n",
    "print(\"\\nDocuments:\")\n",
    "print(f\"  Min: {min(doc_lengths)}\")\n",
    "print(f\"  Max: {max(doc_lengths)}\")\n",
    "print(f\"  Mean: {np.mean(doc_lengths):.1f}\")\n",
    "print(f\"  Median: {np.median(doc_lengths):.0f}\")\n",
    "\n",
    "print(\"\\nOptions:\")\n",
    "print(f\"  Min: {min(option_lengths)}\")\n",
    "print(f\"  Max: {max(option_lengths)}\")\n",
    "print(f\"  Mean: {np.mean(option_lengths):.1f}\")\n",
    "print(f\"  Median: {np.median(option_lengths):.0f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Interactive Sample Browser"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from ipywidgets import interact, IntSlider, Output\n",
    "from IPython.display import display, HTML\n",
    "\n",
    "def display_instance(index):\n",
    "    \"\"\"Display a specific instance with formatting\"\"\"\n",
    "    inst = instances[index]\n",
    "    \n",
    "    html = f\"\"\"\n",
    "    <div style=\"border: 2px solid #3498db; padding: 20px; border-radius: 10px; background-color: #f8f9fa;\">\n",
    "        <h3 style=\"color: #2c3e50;\">📌 Topic ID: {inst.topic_id}</h3>\n",
    "        \n",
    "        <div style=\"margin: 15px 0; padding: 10px; background-color: #fff; border-left: 4px solid #3498db;\">\n",
    "            <h4 style=\"color: #34495e;\">❓ Question:</h4>\n",
    "            <p>{inst.question}</p>\n",
    "        </div>\n",
    "        \n",
    "        <div style=\"margin: 15px 0;\">\n",
    "            <h4 style=\"color: #34495e;\">📝 Options:</h4>\n",
    "            <ul style=\"list-style-type: none; padding-left: 0;\">\n",
    "    \"\"\"\n",
    "    \n",
    "    options = inst.get_options_dict()\n",
    "    colors = {'A': '#3498db', 'B': '#e74c3c', 'C': '#2ecc71', 'D': '#f39c12'}\n",
    "    \n",
    "    for key, value in options.items():\n",
    "        is_answer = key in inst.get_answer_list()\n",
    "        style = f\"background-color: {colors[key]}; color: white;\" if is_answer else \"\"\n",
    "        html += f\"\"\"<li style=\"margin: 5px 0; padding: 10px; {style} border-radius: 5px;\">\n",
    "                    <strong>{key}.</strong> {value}\n",
    "                    {'✅' if is_answer else ''}\n",
    "                   </li>\"\"\"\n",
    "    \n",
    "    html += f\"\"\"\n",
    "            </ul>\n",
    "        </div>\n",
    "        \n",
    "        <div style=\"margin: 15px 0; padding: 10px; background-color: #d5f4e6; border-radius: 5px;\">\n",
    "            <h4 style=\"color: #27ae60;\">✅ Correct Answer: {inst.answer}</h4>\n",
    "        </div>\n",
    "        \n",
    "        <div style=\"margin: 15px 0;\">\n",
    "            <h4 style=\"color: #34495e;\">📄 Documents ({len(inst.docs)} total):</h4>\n",
    "    \"\"\"\n",
    "    \n",
    "    for i, doc in enumerate(inst.docs[:3], 1):  # Show first 3 docs\n",
    "        title = doc.get('title', 'No title')\n",
    "        content = doc.get('content', '')[:200]\n",
    "        html += f\"\"\"\n",
    "        <div style=\"margin: 10px 0; padding: 10px; background-color: #fff; border-left: 3px solid #95a5a6;\">\n",
    "            <strong>Document {i}:</strong> {title}<br>\n",
    "            <small style=\"color: #7f8c8d;\">{content}...</small>\n",
    "        </div>\n",
    "        \"\"\"\n",
    "    \n",
    "    if len(inst.docs) > 3:\n",
    "        html += f\"<p style='color: #7f8c8d;'><em>... and {len(inst.docs) - 3} more documents</em></p>\"\n",
    "    \n",
    "    html += \"</div></div>\"\n",
    "    \n",
    "    display(HTML(html))\n",
    "\n",
    "# Create interactive slider\n",
    "interact(display_instance, index=IntSlider(min=0, max=len(instances)-1, step=1, value=0, description='Instance:'));"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 12. Correlation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create DataFrame for correlation analysis\n",
    "analysis_data = []\n",
    "\n",
    "for inst in instances:\n",
    "    if inst.answer:\n",
    "        analysis_data.append({\n",
    "            'num_docs': len(inst.docs),\n",
    "            'question_length': len(inst.question.split()),\n",
    "            'avg_doc_length': np.mean([len(doc.get('content', '').split()) for doc in inst.docs]) if inst.docs else 0,\n",
    "            'num_correct_options': len(inst.get_answer_list()),\n",
    "            'has_insufficient_info': 1 if 'D' in inst.answer else 0\n",
    "        })\n",
    "\n",
    "df = pd.DataFrame(analysis_data)\n",
    "\n",
    "# Correlation matrix\n",
    "corr = df.corr()\n",
    "\n",
    "fig = go.Figure(data=go.Heatmap(\n",
    "    z=corr.values,\n",
    "    x=corr.columns,\n",
    "    y=corr.columns,\n",
    "    colorscale='RdBu',\n",
    "    zmid=0,\n",
    "    text=corr.values,\n",
    "    texttemplate='%{text:.2f}',\n",
    "    textfont={\"size\": 10}\n",
    "))\n",
    "\n",
    "fig.update_layout(\n",
    "    title='Feature Correlation Matrix',\n",
    "    height=500,\n",
    "    width=700\n",
    ")\n",
    "\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 13. Export Sample Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create sample for manual inspection\n",
    "sample_size = 10\n",
    "sample_instances = instances[:sample_size]\n",
    "\n",
    "sample_data = []\n",
    "for inst in sample_instances:\n",
    "    sample_data.append({\n",
    "        'topic_id': inst.topic_id,\n",
    "        'question': inst.question[:100] + '...',\n",
    "        'answer': inst.answer,\n",
    "        'num_docs': len(inst.docs),\n",
    "        'has_D': 'D' in inst.answer\n",
    "    })\n",
    "\n",
    "sample_df = pd.DataFrame(sample_data)\n",
    "display(sample_df)\n",
    "\n",
    "# Save to CSV\n",
    "output_file = '../outputs/reports/sample_data.csv'\n",
    "sample_df.to_csv(output_file, index=False)\n",
    "print(f\"\\n✓ Sample data saved to {output_file}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 14. Summary Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate comprehensive summary\n",
    "print(\"=\" * 80)\n",
    "print(\"AER DATASET SUMMARY REPORT\")\n",
    "print(\"=\" * 80)\n",
    "\n",
    "print(\"\\n📊 DATASET OVERVIEW\")\n",
    "print(f\"  Total Instances: {len(instances)}\")\n",
    "print(f\"  Total Documents: {sum(len(inst.docs) for inst in instances)}\")\n",
    "print(f\"  Avg Documents per Instance: {np.mean([len(inst.docs) for inst in instances]):.2f}\")\n",
    "\n",
    "print(\"\\n📋 ANSWER PATTERNS\")\n",
    "print(f\"  Single Answer: {single_answers} ({single_answers/len(instances)*100:.1f}%)\")\n",
    "print(f\"  Multiple Answers: {multiple_answers} ({multiple_answers/len(instances)*100:.1f}%)\")\n",
    "print(f\"  Insufficient Info (D only): {has_d_only} ({has_d_only/len(instances)*100:.1f}%)\")\n",
    "\n",
    "print(\"\\n📏 TEXT STATISTICS\")\n",
    "print(f\"  Avg Question Length: {np.mean(question_lengths):.1f} words\")\n",
    "print(f\"  Avg Document Length: {np.mean(doc_lengths):.1f} words\")\n",
    "print(f\"  Avg Option Length: {np.mean(option_lengths):.1f} words\")\n",
    "\n",
    "print(\"\\n🔤 OPTION DISTRIBUTION\")\n",
    "for opt, count in zip(['A', 'B', 'C', 'D'], counts):\n",
    "    print(f\"  Option {opt}: {count} ({count/total*100:.1f}%)\")\n",
    "\n",
    "print(\"\\n\" + \"=\" * 80)\n",
    "print(\"✓ Exploration complete!\")\n",
    "print(\"=\" * 80)"
   ]
  },
  {
   "cell_type":
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\" * 80)\n",
    "print(\"✓ Exploration complete!\")\n",
    "print(\"=\" * 80)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 15. Advanced Analysis: Answer Complexity"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze relationship between document count and answer complexity\n",
    "complexity_data = []\n",
    "\n",
    "for inst in instances:\n",
    "    if inst.answer:\n",
    "        complexity_data.append({\n",
    "            'num_docs': len(inst.docs),\n",
    "            'num_answers': len(inst.get_answer_list()),\n",
    "            'has_insufficient': 'D' in inst.answer\n",
    "        })\n",
    "\n",
    "complexity_df = pd.DataFrame(complexity_data)\n",
    "\n",
    "# Box plot: number of documents vs number of correct answers\n",
    "fig = px.box(\n",
    "    complexity_df, \n",
    "    x='num_answers', \n",
    "    y='num_docs',\n",
    "    title='Document Count vs Answer Complexity',\n",
    "    labels={'num_answers': 'Number of Correct Answers', 'num_docs': 'Number of Documents'},\n",
    "    color='num_answers',\n",
    "    color_discrete_sequence=px.colors.qualitative.Set2\n",
    ")\n",
    "\n",
    "fig.update_layout(height=500, showlegend=False)\n",
    "fig.show()\n",
    "\n",
    "# Statistics by answer complexity\n",
    "print(\"\\nDocument Count by Answer Complexity:\")\n",
    "for num_ans in sorted(complexity_df['num_answers'].unique()):\n",
    "    subset = complexity_df[complexity_df['num_answers'] == num_ans]['num_docs']\n",
    "    print(f\"  {num_ans} answer(s): Mean={subset.mean():.2f}, Median={subset.median():.0f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 16. Word Cloud Visualization"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from wordcloud import WordCloud\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "# Combine all questions\n",
    "all_questions = ' '.join([inst.question for inst in instances])\n",
    "\n",
    "# Create word cloud\n",
    "wordcloud = WordCloud(\n",
    "    width=1200, \n",
    "    height=600, \n",
    "    background_color='white',\n",
    "    colormap='viridis',\n",
    "    max_words=100\n",
    ").generate(all_questions)\n",
    "\n",
    "plt.figure(figsize=(15, 8))\n",
    "plt.imshow(wordcloud, interpolation='bilinear')\n",
    "plt.axis('off')\n",
    "plt.title('Most Common Words in Questions', fontsize=20, fontweight='bold', pad=20)\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"✓ Word cloud generated from all questions\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 17. Topic Distribution (if available)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Count unique topics\n",
    "topic_ids = [inst.topic_id for inst in instances]\n",
    "topic_counts = Counter(topic_ids)\n",
    "\n",
    "print(f\"Total unique topics: {len(topic_counts)}\")\n",
    "print(f\"Topics appearing more than once: {sum(1 for count in topic_counts.values() if count > 1)}\")\n",
    "\n",
    "# Distribution of topic frequencies\n",
    "frequency_dist = Counter(topic_counts.values())\n",
    "\n",
    "fig = go.Figure([\n",
    "    go.Bar(\n",
    "        x=list(frequency_dist.keys()),\n",
    "        y=list(frequency_dist.values()),\n",
    "        text=list(frequency_dist.values()),\n",
    "        textposition='auto',\n",
    "        marker_color='teal'\n",
    "    )\n",
    "])\n",
    "\n",
    "fig.update_layout(\n",
    "    title='Topic Frequency Distribution',\n",
    "    xaxis_title='Number of Instances per Topic',\n",
    "    yaxis_title='Number of Topics',\n",
    "    height=400\n",
    ")\n",
    "\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 18. Filter and Search"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def search_instances(keyword, search_in='question', max_results=5):\n",
    "    \"\"\"\n",
    "    Search for instances containing a keyword\n",
    "    \n",
    "    Args:\n",
    "        keyword: Search term\n",
    "        search_in: 'question', 'options', or 'all'\n",
    "        max_results: Maximum number of results to display\n",
    "    \"\"\"\n",
    "    results = []\n",
    "    \n",
    "    for inst in instances:\n",
    "        found = False\n",
    "        \n",
    "        if search_in in ['question', 'all']:\n",
    "            if keyword.lower() in inst.question.lower():\n",
    "                found = True\n",
    "        \n",
    "        if search_in in ['options', 'all'] and not found:\n",
    "            for opt in inst.get_options_list():\n",
    "                if keyword.lower() in opt.lower():\n",
    "                    found = True\n",
    "                    break\n",
    "        \n",
    "        if found:\n",
    "            results.append(inst)\n",
    "        \n",
    "        if len(results) >= max_results:\n",
    "            break\n",
    "    \n",
    "    print(f\"Found {len(results)} instances containing '{keyword}':\\n\")\n",
    "    \n",
    "    for i, inst in enumerate(results, 1):\n",
    "        print(f\"{i}. Topic ID: {inst.topic_id}\")\n",
    "        print(f\"   Question: {inst.question[:100]}...\")\n",
    "        print(f\"   Answer: {inst.answer}\")\n",
    "        print()\n",
    "\n",
    "# Example searches\n",
    "search_instances('climate', search_in='all', max_results=3)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 19. Export Summary Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "\n",
    "# Create comprehensive statistics dictionary\n",
    "summary_stats = {\n",
    "    'dataset_overview': {\n",
    "        'total_instances': len(instances),\n",
    "        'total_documents': sum(len(inst.docs) for inst in instances),\n",
    "        'avg_docs_per_instance': float(np.mean([len(inst.docs) for inst in instances])),\n",
    "        'unique_topics': len(set(inst.topic_id for inst in instances))\n",
    "    },\n",
    "    'answer_distribution': {\n",
    "        'single_answer_count': int(single_answers),\n",
    "        'multiple_answer_count': int(multiple_answers),\n",
    "        'insufficient_info_only': int(has_d_only),\n",
    "        'top_10_answers': [(ans, int(count)) for ans, count in answer_counts.most_common(10)]\n",
    "    },\n",
    "    'text_statistics': {\n",
    "        'question_length': {\n",
    "            'min': int(min(question_lengths)),\n",
    "            'max': int(max(question_lengths)),\n",
    "            'mean': float(np.mean(question_lengths)),\n",
    "            'median': float(np.median(question_lengths))\n",
    "        },\n",
    "        'document_length': {\n",
    "            'min': int(min(doc_lengths)),\n",
    "            'max': int(max(doc_lengths)),\n",
    "            'mean': float(np.mean(doc_lengths)),\n",
    "            'median': float(np.median(doc_lengths))\n",
    "        }\n",
    "    },\n",
    "    'option_frequencies': {\n",
    "        opt: int(count) for opt, count in zip(['A', 'B', 'C', 'D'], counts)\n",
    "    }\n",
    "}\n",
    "\n",
    "# Save to JSON\n",
    "output_file = '../outputs/reports/exploration_summary.json'\n",
    "with open(output_file, 'w') as f:\n",
    "    json.dump(summary_stats, f, indent=2)\n",
    "\n",
    "print(f\"✓ Summary statistics saved to {output_file}\")\n",
    "print(\"\\nSummary contents:\")\n",
    "print(json.dumps(summary_stats, indent=2))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 20. Recommendations for Model Development"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"RECOMMENDATIONS FOR MODEL DEVELOPMENT\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "print(\"\\n🎯 Key Insights:\")\n",
    "print(f\"\\n1. Answer Complexity:\")\n",
    "print(f\"   - {(multiple_answers/len(instances)*100):.1f}% of instances have multiple correct answers\")\n",
    "print(f\"   - Model should handle multi-label classification\")\n",
    "\n",
    "print(f\"\\n2. Insufficient Information Cases:\")\n",
    "print(f\"   - {(has_d_only/len(instances)*100):.1f}% are purely 'insufficient information'\")\n",
    "print(f\"   - Model needs to learn when NOT to make a determination\")\n",
    "\n",
    "print(f\"\\n3. Document Volume:\")\n",
    "print(f\"   - Average {np.mean([len(inst.docs) for inst in instances]):.1f} documents per instance\")\n",
    "print(f\"   - Retrieval/ranking strategy is important\")\n",
    "print(f\"   - Consider document relevance scoring\")\n",
    "\n",
    "print(f\"\\n4. Text Length:\")\n",
    "print(f\"   - Average document: {np.mean(doc_lengths):.0f} words\")\n",
    "print(f\"   - Need efficient context management for LLMs\")\n",
    "print(f\"   - Consider chunking or summarization for long documents\")\n",
    "\n",
    "print(f\"\\n💡 Suggested Approaches:\")\n",
    "print(\"\\n1. Retrieval-Augmented Generation (RAG)\")\n",
    "print(\"   - Use embeddings to find most relevant documents\")\n",
    "print(\"   - Rerank documents before passing to LLM\")\n",
    "\n",
    "print(\"\\n2. Chain-of-Thought Prompting\")\n",
    "print(\"   - Guide model to reason about evidence\")\n",
    "print(\"   - Explicitly ask to evaluate each option\")\n",
    "\n",
    "print(\"\\n3. Few-Shot Learning\")\n",
    "print(\"   - Include examples with multiple answers\")\n",
    "print(\"   - Include examples with 'insufficient information'\")\n",
    "\n",
    "print(\"\\n4. Ensemble Methods\")\n",
    "print(\"   - Combine multiple models/prompts\")\n",
    "print(\"   - Use voting or confidence-based selection\")\n",
    "\n",
    "print(\"\\n5. Confidence Calibration\")\n",
    "print(\"   - Model confidence scores for each option\")\n",
    "print(\"   - Use threshold for 'insufficient information'\")\n",
    "\n",
    "print(\"\\n\" + \"=\"*80)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}